# How to Use Generators and yield in Python
https://realpython.com/introduction-to-python-generators/

### Using Generators

#### Example 1: Reading Large Files

In [ ]:
# This fails if the file size is larger then the available memory
def csv_reader(file_name):
    file = open(file_name) # returns generator object
    result = file.read().split("\n") # loads everithing into memory at once
    return result

# generator function
def csv_reader_large_files(file_name):
    for row in open(file_name, "r"):
        yield row

csv_gen = csv_reader("some_csv.txt")
csv_gen = (row for row in open("some_csv.txt")) # generator comprehension version
row_cont = 0

for row in csv_gen:
    row_count += 1

print(f"Row count is {row_count}")

### Example 2: Generating an Infinite Sequence


In [2]:
a = range(5)
print(list(a))

[0, 1, 2, 3, 4]


In [ ]:
def infinite_sequence():
    num = 0
    while True:
        yield num
        num += 1

for i in infinite_sequence():
    print(i, end=" ")
    if len(str(i)) == 5:
        infinite_sequence().close()

gen = infinite_sequence()
print(next(gen))
print(next(gen))
print(next(gen))

0 

TypeError: object of type 'int' has no len()

### Example 3: Reading techcrunch.csv with Generators

O CSV tem ~1400 linhas de rodadas de financiamento de startups (TechCrunch dataset).  
O objetivo é processar o arquivo **sem carregar tudo na memória** — cada row é produzida sob demanda.

In [6]:
import csv
from pathlib import Path

CSV_PATH = Path("techcrunch.csv")


def csv_rows(path: Path) -> "Generator[dict[str, str], None, None]":
    """Yields one parsed row at a time — O(1) memória independente do tamanho do arquivo."""
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            yield row


# Pipeline de generators — nenhum dado é materializado até o for final
rows = csv_rows(CSV_PATH)

# Filtra apenas rodadas "series-a" com valor acima de $5M (USD)
big_series_a = (
    row for row in rows
    if row["round"] == "a" and row["raisedCurrency"] == "USD" and int(row["raisedAmt"]) > 5_000_000
)

print(f"{'Empresa':<30} {'Cidade':<20} {'Valor':>15}")
print("-" * 67)

for row in big_series_a:
    print(f"{row['company']:<31} {row['city']:<20} ${int(row['raisedAmt']):>14,.0f}")

Empresa                        Cidade                         Valor
-------------------------------------------------------------------
LifeLock                        Tempe                $     6,000,000
Infusionsoft                    Gilbert              $     9,000,000
Facebook                        Palo Alto            $    12,700,000
Gizmoz                          Menlo Park           $     6,300,000
Slacker                         San Diego            $    13,500,000
Lala                            Palo Alto            $     9,000,000
Powerset                        San Francisco        $    12,500,000
JotSpot                         Palo Alto            $     5,200,000
Prosper                         San Francisco        $     7,500,000
Google                          Mountain View        $    25,000,000
Ustream                         Mountain View        $    11,100,000
GizmoProject                    San Diego            $     6,000,000
Adap.tv                         San 

In [2]:
# Agregação lazy: total levantado por categoria, sem Pandas
from collections import defaultdict

totals: dict[str, int] = defaultdict(int)

for row in csv_rows(CSV_PATH):
    if row["raisedCurrency"] == "USD":
        totals[row["category"]] += int(row["raisedAmt"])

# top 5 categorias
top5 = sorted(totals.items(), key=lambda x: x[1], reverse=True)[:5]

print(f"{'Categoria':<25} {'Total USD':>20}")
print("-" * 47)
for category, total in top5:
    print(f"{category:<25} ${total:>19,.0f}")

Categoria                            Total USD
-----------------------------------------------
web                       $     11,753,474,750
software                  $      1,017,942,000
hardware                  $        824,500,000
                          $        373,300,000
mobile                    $        323,020,000
